# BioNER experiment runner (Colab / Kaggle)

Trains the experiments in `configs/` on a GPU and writes one JSON per (experiment, seed, corpus) under `results/`.
Nothing is estimated: every number in `docs/RESULTS.md` comes from files this notebook produces.

**Before you start:** Runtime / Accelerator -> GPU (T4 is fine). On Kaggle: Settings -> Accelerator -> GPU, Internet on.
No tokens or API keys are needed for the training experiments (all models are public).

**How long (rough estimate, not measured):** one experiment x one seed over all five corpora is roughly 30 to 40 minutes on a T4
(`mtl_pcgrad` roughly 2 to 3 times that, it runs in fp32). Run the tiers below in separate sessions; finished results are
skipped when you rerun, so a disconnect only loses the run in progress.

| Tier | `EXPERIMENTS` | Why |
|---|---|---|
| 1 | `stl_biobert`, `mtl_proportional`, `mtl_temp2` | the core single-task vs multi-task comparison |
| 2 | `mtl_uniform`, `stl_pubmedbert` | sampling ablation, newer encoder |
| 3 | `stl_biolinkbert`, `mtl_pcgrad` | optional extras |

When done, download `results.zip` and put its contents in the repo's `results/` folder (or hand it to Claude), then run
`python -m bioner report`.

In [ ]:
import os, subprocess, sys

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/working")
print("colab:", IN_COLAB, "kaggle:", IN_KAGGLE)
print(
    subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout
    or "NO GPU - switch the runtime to GPU"
)

In [ ]:
# ---- settings you may change -------------------------------------------------------------
REPO = "https://github.com/VanshikaGurbani/Multi-BIONER"
BRANCH = "upgrade/rigorous-bioner"  # use "main" once the PR is merged
EXPERIMENTS = ["stl_biobert", "mtl_proportional", "mtl_temp2"]  # see the tier table above
SEEDS = [13, 42, 2024]  # keep >= 3 seeds for mean +- std
# -------------------------------------------------------------------------------------------

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")  # results survive a disconnect
    OUT_DIR = "/content/drive/MyDrive/bioner_results"
elif IN_KAGGLE:
    OUT_DIR = "/kaggle/working/results"
else:
    OUT_DIR = os.path.abspath("results")
os.makedirs(OUT_DIR, exist_ok=True)
print("results go to", OUT_DIR)

In [ ]:
WORK = "/content" if IN_COLAB else ("/kaggle/working" if IN_KAGGLE else os.getcwd())
os.chdir(WORK)
if not os.path.exists("Multi-BIONER"):
    !git clone --branch {BRANCH} {REPO}
os.chdir("Multi-BIONER")
!git log --oneline -1
!pip install -q -r requirements-colab.txt
!pip install -q -e . --no-deps

In [ ]:
!python -m bioner prepare-data

In [ ]:
for exp in EXPERIMENTS:
    print("=" * 30, exp, "=" * 30, flush=True)
    seeds = " ".join(map(str, SEEDS))
    !python -m bioner train --config configs/{exp}.yaml --out-dir {OUT_DIR} --seeds {seeds}

In [ ]:
# Summary of everything in OUT_DIR so far (also what you can paste back to Claude)
!python -m bioner report --results-dir {OUT_DIR} --out /tmp/RESULTS_preview.md --readme ""
print(open("/tmp/RESULTS_preview.md").read())

In [ ]:
import shutil

shutil.make_archive("/tmp/results", "zip", OUT_DIR)
print("zip size MB:", os.path.getsize("/tmp/results.zip") / 1e6)
if IN_COLAB:
    from google.colab import files

    files.download("/tmp/results.zip")
elif IN_KAGGLE:
    shutil.copy("/tmp/results.zip", "/kaggle/working/results.zip")
    print(
        "Saved /kaggle/working/results.zip - use 'Save Version' and download it from the Output tab."
    )